<a href="https://colab.research.google.com/github/Sgnsilv/ResearchAtlas/blob/5-machine-learning-implementar-pipeline-de-embeddings-sem%C3%A2nticos/notebooks/01_embeddings_semanticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResearchAtlas — Embeddings Semânticos

**Issue #05 — Pipeline de embeddings semânticos**

## Objetivo

Construir representações vetoriais dos artigos científicos utilizando modelos Sentence Transformers, preservando a correspondência entre os embeddings e os identificadores OpenAlex.

## Dataset

- Fonte: OpenAlex
- Artigos: 5.000
- Áreas: Medicina, Astronomia, Biologia, Clima e Engenharia
- Período: 2020–2025
- Versão: dataset selecionado v1
- Semente: 42

## Etapas

1. Configuração do ambiente
2. Carregamento e validação do dataset
3. Preparação textual
4. Seleção do modelo semântico
5. Geração dos embeddings
6. Validação e persistência dos artefatos


In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Sgnsilv/ResearchAtlas.git"
BRANCH = "5-machine-learning-implementar-pipeline-de-embeddings-semânticos"
PROJECT_DIR = Path("/content/ResearchAtlas")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_DIR),
        ],
        check=True,
    )

sys.path.insert(0, str(PROJECT_DIR / "src"))

print("Repositório:", PROJECT_DIR)
print("Branch:", BRANCH)


Repositório: /content/ResearchAtlas
Branch: 5-machine-learning-implementar-pipeline-de-embeddings-semânticos


In [2]:
%pip install -q pandas pyarrow numpy sentence-transformers

import torch
import pandas as pd
import numpy as np

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU indisponível; execução em CPU.")


PyTorch: 2.14.1+cu130
Dispositivo: cuda
GPU: Tesla T4


In [3]:
import hashlib
import json

DATASET_PATH = (
    PROJECT_DIR / "data/processed/dataset_selected.parquet"
)

EXPECTED_SHA256 = (
    "533fe010fee10dcc87e4ec196886e8a8d9ec6e17f0f4f3d507dc0cf278c79389"
)

with DATASET_PATH.open("rb") as file:
    dataset_hash = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

assert dataset_hash == EXPECTED_SHA256, (
    "O dataset não corresponde à versão v1."
)

df = pd.read_parquet(DATASET_PATH)

assert len(df) == 5000
assert df["openalex_id"].is_unique

print("=== DATASET CARREGADO ===")
print("Artigos:", len(df))
print("Colunas:", len(df.columns))
print("IDs únicos:", df["openalex_id"].nunique())
print("Integridade SHA-256: APROVADA")

display(
    df[
        [
            "openalex_id",
            "title",
            "sampling_area",
            "publication_year",
        ]
    ].head()
)

=== DATASET CARREGADO ===
Artigos: 5000
Colunas: 21
IDs únicos: 5000
Integridade SHA-256: APROVADA


,openalex_id,title,sampling_area,publication_year
0,https://openalex.org/W3030030520,Gender imbalance in medical imaging datasets p...,Medicina,2020
1,https://openalex.org/W3048540441,Artificial intelligence and deep learning in o...,Medicina,2020
2,https://openalex.org/W3045804302,Making a big impact with small datasets using ...,Medicina,2020
3,https://openalex.org/W3083817441,Development of machine learning-based clinical...,Medicina,2020
4,https://openalex.org/W3109556460,Artificial intelligence for automatic cerebral...,Medicina,2020


In [4]:
import re

def prepare_text(title: str, abstract: str) -> str:
    """Prepara título e resumo para representação semântica."""
    title = re.sub(r"\s+", " ", str(title)).strip()
    abstract = re.sub(r"\s+", " ", str(abstract)).strip()

    return f"{title}. {abstract}"


texts = [
    prepare_text(title, abstract)
    for title, abstract in zip(
        df["title"],
        df["abstract"],
    )
]

df["embedding_text"] = texts

df["text_word_count"] = (
    df["embedding_text"]
    .str.split()
    .str.len()
)

print("=== PREPARAÇÃO TEXTUAL ===")
print("Textos preparados:", len(texts))
print("Textos vazios:", sum(not text.strip() for text in texts))

print("\n=== COMPRIMENTO DOS TEXTOS ===")
print(df["text_word_count"].describe().round(2))

print("\n=== EXEMPLO ===")
print(texts[0][:1000])


=== PREPARAÇÃO TEXTUAL ===
Textos preparados: 5000
Textos vazios: 0

=== COMPRIMENTO DOS TEXTOS ===
count    5000.00
mean      251.93
std       223.27
min        12.00
25%       181.00
50%       225.00
75%       270.00
max      4219.00
Name: text_word_count, dtype: float64

=== EXEMPLO ===
Gender imbalance in medical imaging datasets produces biased classifiers for computer-aided diagnosis. Artificial intelligence (AI) systems for computer-aided diagnosis and image-based screening are being adopted worldwide by medical institutions. In such a context, generating fair and unbiased classifiers becomes of paramount importance. The research community of medical image computing is making great efforts in developing more accurate algorithms to assist medical doctors in the difficult task of disease diagnosis. However, little attention is paid to the way databases are collected and how this may influence the performance of AI systems. Our study sheds light on the importance of gender balance 

In [5]:
flags = df["quality_flags"].apply(json.loads)

df["has_quality_alert"] = flags.apply(bool)

print("=== QUALIDADE DAS ENTRADAS ===")
print("Total:", len(df))
print("Com alerta:", int(df["has_quality_alert"].sum()))
print("Sem alerta:", int((~df["has_quality_alert"]).sum()))

print("\n=== ALERTAS POR ÁREA ===")
print(
    df.groupby("sampling_area")["has_quality_alert"]
    .agg(["sum", "count"])
)

print("\n=== EXEMPLOS SINALIZADOS ===")
display(
    df.loc[
        df["has_quality_alert"],
        [
            "title",
            "abstract_word_count",
            "quality_flags",
        ],
    ].head(10)
)


=== QUALIDADE DAS ENTRADAS ===
Total: 5000
Com alerta: 195
Sem alerta: 4805

=== ALERTAS POR ÁREA ===
               sum  count
sampling_area            
Astronomia      22   1000
Biologia        81   1000
Clima           21   1000
Engenharia      16   1000
Medicina        55   1000

=== EXEMPLOS SINALIZADOS ===


,title,abstract_word_count,quality_flags
39,Use of artificial intelligence-based computer ...,803,"[""long_abstract""]"
41,Iris Localization using the Pupil Center Point...,9,"[""short_abstract""]"
54,Noise and Signal Characteristics of Deep Learn...,635,"[""long_abstract""]"
80,Rapid triage for COVID-19 using routine clinic...,591,"[""long_abstract""]"
82,Using Artificial Intelligence to Identify Surg...,42,"[""short_abstract""]"
85,Advanced Quantitative MSK Imaging,652,"[""long_abstract""]"
96,Artificial intelligence in reproductive medicine,616,"[""long_abstract""]"
100,What Are the 510(k) Requirements for AI Medica...,19,"[""short_abstract""]"
114,Artificial intelligence modeling of vascular c...,1223,"[""long_abstract""]"
144,Musculoskeletal Imaging Applications of Artifi...,519,"[""long_abstract""]"


In [6]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

tokenizer = model.tokenizer
max_tokens = model.max_seq_length

print("=== MODELO SEMÂNTICO ===")
print("Modelo:", MODEL_NAME)
print("Dispositivo:", model.device)
print("Dimensões:", model.get_sentence_embedding_dimension())
print("Limite de tokens:", max_tokens)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== MODELO SEMÂNTICO ===
Modelo: sentence-transformers/all-MiniLM-L6-v2
Dispositivo: cuda:0
Dimensões: 384
Limite de tokens: 256


/tmp/ipykernel_1741/1902755495.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensões:", model.get_sentence_embedding_dimension())


In [7]:
import numpy as np

token_counts = []

for start in range(0, len(texts), 128):
    batch = texts[start:start + 128]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )

    token_counts.extend(
        len(input_ids)
        for input_ids in encoded["input_ids"]
    )

df["token_count"] = token_counts
df["would_truncate"] = df["token_count"] > max_tokens

print("=== COMPRIMENTO DOS TEXTOS EM TOKENS ===")
print(df["token_count"].describe().round(2))

print("\n=== TRUNCAMENTO ===")
print("Limite:", max_tokens)
print("Artigos que excedem o limite:", int(df["would_truncate"].sum()))
print("Percentual:", f"{df['would_truncate'].mean():.2%}")

print("\n=== TRUNCAMENTO POR ÁREA ===")
print(
    df.groupby("sampling_area")["would_truncate"]
    .agg(["sum", "mean"])
    .round(3)
)

print("\n=== TRUNCAMENTO E QUALIDADE TEXTUAL ===")
print(
    pd.crosstab(
        df["has_quality_alert"],
        df["would_truncate"],
    )
)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (274 > 256). Running this sequence through the model will result in indexing errors


=== COMPRIMENTO DOS TEXTOS EM TOKENS ===
count    5000.00
mean      352.79
std       308.63
min        26.00
25%       245.00
50%       312.50
75%       385.00
max      5200.00
Name: token_count, dtype: float64

=== TRUNCAMENTO ===
Limite: 256
Artigos que excedem o limite: 3552
Percentual: 71.04%

=== TRUNCAMENTO POR ÁREA ===
               sum   mean
sampling_area            
Astronomia     696  0.696
Biologia       747  0.747
Clima          765  0.765
Engenharia     587  0.587
Medicina       757  0.757

=== TRUNCAMENTO E QUALIDADE TEXTUAL ===
would_truncate     False  True 
has_quality_alert              
False               1382   3423
True                  66    129


In [8]:
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE,
)

bge_tokenizer = bge_model.tokenizer
bge_max_tokens = bge_model.max_seq_length

print("=== MODELO ALTERNATIVO ===")
print("Modelo:", BGE_MODEL_NAME)
print("Dispositivo:", bge_model.device)
print("Dimensões:", bge_model.get_embedding_dimension())
print("Limite de tokens:", bge_max_tokens)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== MODELO ALTERNATIVO ===
Modelo: BAAI/bge-small-en-v1.5
Dispositivo: cuda:0
Dimensões: 384
Limite de tokens: 512


In [9]:
bge_token_counts = []

for start in range(0, len(texts), 128):
    batch = texts[start:start + 128]

    encoded = bge_tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        verbose=False,
    )

    bge_token_counts.extend(
        len(ids) for ids in encoded["input_ids"]
    )

df["bge_token_count"] = bge_token_counts
df["bge_would_truncate"] = (
    df["bge_token_count"] > bge_max_tokens
)

comparison = pd.DataFrame({
    "Modelo": [
        "MiniLM-L6-v2",
        "BGE-small-en-v1.5",
    ],
    "Limite": [
        max_tokens,
        bge_max_tokens,
    ],
    "Artigos truncados": [
        int(df["would_truncate"].sum()),
        int(df["bge_would_truncate"].sum()),
    ],
    "Percentual": [
        round(df["would_truncate"].mean() * 100, 2),
        round(df["bge_would_truncate"].mean() * 100, 2),
    ],
})

print("=== COMPARAÇÃO DE MODELOS ===")
display(comparison)

print("\n=== BGE: TRUNCAMENTO POR ÁREA ===")
display(
    df.groupby("sampling_area")["bge_would_truncate"]
    .agg(["sum", "mean"])
    .round(3)
)


=== COMPARAÇÃO DE MODELOS ===


,Modelo,Limite,Artigos truncados,Percentual
0,MiniLM-L6-v2,256,3552,71.04
1,BGE-small-en-v1.5,512,412,8.24



=== BGE: TRUNCAMENTO POR ÁREA ===


,sum,mean
sampling_area,,
Astronomia,51,0.051
Biologia,143,0.143
Clima,67,0.067
Engenharia,28,0.028
Medicina,123,0.123


In [10]:
import math

CHUNK_SIZE = 510
CHUNK_OVERLAP = 64
CHUNK_STEP = CHUNK_SIZE - CHUNK_OVERLAP

def estimate_chunks(token_count: int) -> int:
    """Estima quantos trechos serão necessários."""
    content_tokens = max(0, token_count - 2)

    if content_tokens <= CHUNK_SIZE:
        return 1

    return 1 + math.ceil(
        (content_tokens - CHUNK_SIZE) / CHUNK_STEP
    )


df["estimated_chunks"] = (
    df["bge_token_count"].apply(estimate_chunks)
)

print("=== SEGMENTAÇÃO ESTIMADA ===")
print("Artigos com um trecho:", int((df["estimated_chunks"] == 1).sum()))
print("Artigos com múltiplos trechos:", int((df["estimated_chunks"] > 1).sum()))
print("Total de trechos:", int(df["estimated_chunks"].sum()))
print("Máximo de trechos por artigo:", int(df["estimated_chunks"].max()))

print("\n=== DISTRIBUIÇÃO DE TRECHOS ===")
print(
    df["estimated_chunks"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== CINCO ARTIGOS MAIS LONGOS ===")
display(
    df.nlargest(5, "bge_token_count")[
        [
            "title",
            "sampling_area",
            "bge_token_count",
            "estimated_chunks",
        ]
    ]
)


=== SEGMENTAÇÃO ESTIMADA ===
Artigos com um trecho: 4588
Artigos com múltiplos trechos: 412
Total de trechos: 5678
Máximo de trechos por artigo: 12

=== DISTRIBUIÇÃO DE TRECHOS ===
estimated_chunks
1     4588
2      330
3       33
4       15
5       11
6        2
7        4
8        1
9        3
10       4
11       7
12       2

=== CINCO ARTIGOS MAIS LONGOS ===


,title,sampling_area,bge_token_count,estimated_chunks
2912,Impacts of heat stress on growth performance a...,Biologia,5200,12
2826,Variants of the GNAI1 gene manifest as Prader–...,Biologia,5001,12
2969,Cohort Profile: The Girona Heart Registry,Biologia,4946,11
2235,The genes of OMIM : A legacy of Victor McKusick,Biologia,4778,11
2157,The Dynamic and Multifaceted Nature of Cardiov...,Biologia,4750,11


In [13]:
SPECIAL_TOKENS = bge_tokenizer.num_special_tokens_to_add(pair=False)

CHUNK_SIZE = bge_max_tokens - SPECIAL_TOKENS
CHUNK_OVERLAP = 64

CLS_ID = bge_tokenizer.cls_token_id
SEP_ID = bge_tokenizer.sep_token_id

assert SPECIAL_TOKENS == 2
assert CLS_ID is not None
assert SEP_ID is not None
assert CHUNK_SIZE > CHUNK_OVERLAP


def split_into_chunks(text: str) -> list[dict]:
    """Divide o texto em trechos preservando os IDs originais."""

    token_ids = bge_tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    )

    if not token_ids:
        raise ValueError("Texto vazio após tokenização")

    chunks = []
    start = 0

    while start < len(token_ids):
        end = min(start + CHUNK_SIZE, len(token_ids))

        content_ids = token_ids[start:end]

        # Sequência BERT: [CLS] + conteúdo + [SEP]
        input_ids = [CLS_ID] + content_ids + [SEP_ID]

        assert len(input_ids) <= bge_max_tokens

        # Desconta a sobreposição na ponderação.
        new_tokens = end - start

        if start > 0:
            new_tokens -= CHUNK_OVERLAP

        chunks.append({
            "input_ids": input_ids,
            "token_weight": max(1, new_tokens),
        })

        if end == len(token_ids):
            break

        start = end - CHUNK_OVERLAP

    return chunks


print("=== CONFIGURAÇÃO DO CHUNKING ===")
print("Limite do modelo:", bge_max_tokens)
print("Tokens especiais:", SPECIAL_TOKENS)
print("Tokens de conteúdo:", CHUNK_SIZE)
print("Sobreposição:", CHUNK_OVERLAP)

longest_index = int(df["bge_token_count"].idxmax())
longest_text = texts[longest_index]

example_chunks = split_into_chunks(longest_text)

print("\n=== TESTE COM O MAIOR ARTIGO ===")
print("Título:", df.loc[longest_index, "title"])
print("Trechos:", len(example_chunks))
print(
    "Tokens por trecho:",
    [len(chunk["input_ids"]) for chunk in example_chunks]
)

assert all(
    len(chunk["input_ids"]) <= bge_max_tokens
    for chunk in example_chunks
)

print("\nSEGMENTAÇÃO DO MAIOR ARTIGO: APROVADA")


=== CONFIGURAÇÃO DO CHUNKING ===
Limite do modelo: 512
Tokens especiais: 2
Tokens de conteúdo: 510
Sobreposição: 64

=== TESTE COM O MAIOR ARTIGO ===
Título: Impacts of heat stress on growth performance and its mitigation in small ruminants
Trechos: 12
Tokens por trecho: [512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 294]

SEGMENTAÇÃO DO MAIOR ARTIGO: APROVADA


In [14]:
from collections import Counter

all_chunks = []
chunk_article_indices = []
chunk_token_weights = []

for article_index, text in enumerate(texts):
    chunks = split_into_chunks(text)

    for chunk in chunks:
        all_chunks.append(chunk["input_ids"])
        chunk_article_indices.append(article_index)
        chunk_token_weights.append(chunk["token_weight"])

chunk_distribution = Counter(chunk_article_indices)

print("=== SEGMENTAÇÃO FINAL ===")
print("Artigos:", len(texts))
print("Total de trechos:", len(all_chunks))
print(
    "Artigos com múltiplos trechos:",
    sum(count > 1 for count in chunk_distribution.values())
)
print("Máximo de trechos:", max(chunk_distribution.values()))
print(
    "Maior comprimento:",
    max(len(chunk) for chunk in all_chunks),
)

assert len(chunk_distribution) == len(texts)
assert len(all_chunks) == len(chunk_article_indices)
assert len(all_chunks) == len(chunk_token_weights)

assert all(
    len(chunk) <= bge_max_tokens
    for chunk in all_chunks
)

print("\nSEGMENTAÇÃO: APROVADA")


=== SEGMENTAÇÃO FINAL ===
Artigos: 5000
Total de trechos: 5678
Artigos com múltiplos trechos: 412
Máximo de trechos: 12
Maior comprimento: 512

SEGMENTAÇÃO: APROVADA


In [15]:
import torch
import torch.nn.functional as F


def encode_token_batch(token_sequences):
    """Gera embeddings usando IDs já tokenizados."""
    input_ids = bge_tokenizer.pad_token_id

    max_length = max(len(ids) for ids in token_sequences)

    padded_ids = []
    attention_masks = []

    for ids in token_sequences:
        padding = max_length - len(ids)

        padded_ids.append(
            ids + [input_ids] * padding
        )

        attention_masks.append(
            [1] * len(ids) + [0] * padding
        )

    features = {
        "input_ids": torch.tensor(
            padded_ids, dtype=torch.long, device=DEVICE
        ),
        "attention_mask": torch.tensor(
            attention_masks, dtype=torch.long, device=DEVICE
        ),
    }

    with torch.inference_mode():
        output = bge_model(features)
        embeddings = output["sentence_embedding"]
        embeddings = F.normalize(embeddings.float(), p=2, dim=1)

    return embeddings.cpu().numpy()


# Seleciona artigos que não precisam de segmentação.
test_indices = [
    i for i, chunks in enumerate(
        df["estimated_chunks"]
    )
    if chunks == 1
][:3]

test_texts = [texts[i] for i in test_indices]

reference_embeddings = bge_model.encode(
    test_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

manual_embeddings = encode_token_batch(
    [
        split_into_chunks(text)[0]["input_ids"]
        for text in test_texts
    ]
)

similarities = (
    reference_embeddings * manual_embeddings
).sum(axis=1)

print("=== VALIDAÇÃO DO ENCODER ===")
print("Similaridades:", similarities)
print("Dimensões:", manual_embeddings.shape)

assert manual_embeddings.shape == (3, 384)
assert (similarities > 0.999).all()

print("\nENCODER: APROVADO")


=== VALIDAÇÃO DO ENCODER ===
Similaridades: [1.        1.0000001 1.0000001]
Dimensões: (3, 384)

ENCODER: APROVADO


In [16]:
import numpy as np
from tqdm.auto import tqdm

BATCH_SIZE = 16
EMBEDDING_DIM = bge_model.get_embedding_dimension()

bge_model.eval()

chunk_embeddings = np.empty(
    (len(all_chunks), EMBEDDING_DIM),
    dtype=np.float32,
)

print("=== GERAÇÃO DOS EMBEDDINGS ===")
print("Modelo:", BGE_MODEL_NAME)
print("Dispositivo:", DEVICE)
print("Trechos:", len(all_chunks))
print("Dimensões:", EMBEDDING_DIM)
print("Batch size:", BATCH_SIZE)

for start in tqdm(
    range(0, len(all_chunks), BATCH_SIZE),
    desc="Gerando embeddings",
):
    end = min(start + BATCH_SIZE, len(all_chunks))

    batch_embeddings = encode_token_batch(
        all_chunks[start:end]
    )

    chunk_embeddings[start:end] = batch_embeddings

print("\n=== VALIDAÇÃO ===")
print("Shape:", chunk_embeddings.shape)
print("Tipo:", chunk_embeddings.dtype)
print("Valores finitos:", np.isfinite(chunk_embeddings).all())

assert chunk_embeddings.shape == (5678, 384)
assert np.isfinite(chunk_embeddings).all()

print("\nEMBEDDINGS DOS TRECHOS: APROVADOS")


=== GERAÇÃO DOS EMBEDDINGS ===
Modelo: BAAI/bge-small-en-v1.5
Dispositivo: cuda
Trechos: 5678
Dimensões: 384
Batch size: 16


Gerando embeddings:   0%|          | 0/355 [00:00<?, ?it/s]


=== VALIDAÇÃO ===
Shape: (5678, 384)
Tipo: float32
Valores finitos: True

EMBEDDINGS DOS TRECHOS: APROVADOS


In [17]:
article_embeddings = np.zeros(
    (len(df), EMBEDDING_DIM),
    dtype=np.float64,
)

article_weights = np.zeros(
    len(df),
    dtype=np.float64,
)

for chunk_idx, article_idx in enumerate(chunk_article_indices):
    weight = chunk_token_weights[chunk_idx]

    article_embeddings[article_idx] += (
        chunk_embeddings[chunk_idx] * weight
    )

    article_weights[article_idx] += weight

assert (article_weights > 0).all()

# Média ponderada por quantidade de tokens novos.
article_embeddings /= article_weights[:, None]

# Normalização L2.
norms = np.linalg.norm(
    article_embeddings,
    axis=1,
    keepdims=True,
)

assert (norms > 0).all()

article_embeddings = (
    article_embeddings / norms
).astype(np.float32)

print("=== EMBEDDINGS POR ARTIGO ===")
print("Shape:", article_embeddings.shape)
print("Tipo:", article_embeddings.dtype)
print("Valores finitos:", np.isfinite(article_embeddings).all())

final_norms = np.linalg.norm(article_embeddings, axis=1)

print("Norma mínima:", final_norms.min())
print("Norma máxima:", final_norms.max())

assert article_embeddings.shape == (5000, 384)
assert np.isfinite(article_embeddings).all()
assert np.allclose(final_norms, 1.0, atol=1e-5)

print("\nAGREGAÇÃO: APROVADA")


=== EMBEDDINGS POR ARTIGO ===
Shape: (5000, 384)
Tipo: float32
Valores finitos: True
Norma mínima: 0.99999994
Norma máxima: 1.0

AGREGAÇÃO: APROVADA


In [18]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/ResearchAtlas/"
    "embeddings/bge-small-en-v1.5-v1"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Diretório:", ARTIFACT_DIR)


Mounted at /content/drive
Diretório: /content/drive/MyDrive/ResearchAtlas/embeddings/bge-small-en-v1.5-v1


In [19]:
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

EMBEDDINGS_PATH = ARTIFACT_DIR / "embeddings.npy"
METADATA_PATH = ARTIFACT_DIR / "metadata.parquet"
MANIFEST_PATH = ARTIFACT_DIR / "manifest.json"

np.save(EMBEDDINGS_PATH, article_embeddings)

metadata = df[
    [
        "openalex_id",
        "title",
        "sampling_area",
        "publication_year",
        "quality_flags",
        "requires_review",
        "bge_token_count",
    ]
].copy()

metadata.insert(
    0,
    "embedding_index",
    np.arange(len(metadata)),
)

metadata["chunk_count"] = [
    chunk_distribution[i] for i in range(len(metadata))
]

metadata.to_parquet(
    METADATA_PATH,
    index=False,
)

with EMBEDDINGS_PATH.open("rb") as file:
    embeddings_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": EXPECTED_SHA256,
    "model": BGE_MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIM,
    "article_count": len(df),
    "chunk_count": len(all_chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "batch_size": BATCH_SIZE,
    "aggregation": "weighted_mean_then_l2_normalization",
    "chunk_normalization": "l2",
    "embedding_dtype": "float32",
    "embedding_file_sha256": embeddings_sha256,
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("=== ARTEFATOS SALVOS ===")
print(EMBEDDINGS_PATH)
print(METADATA_PATH)
print(MANIFEST_PATH)

print("\nSHA-256 dos embeddings:")
print(embeddings_sha256)


=== ARTEFATOS SALVOS ===
/content/drive/MyDrive/ResearchAtlas/embeddings/bge-small-en-v1.5-v1/embeddings.npy
/content/drive/MyDrive/ResearchAtlas/embeddings/bge-small-en-v1.5-v1/metadata.parquet
/content/drive/MyDrive/ResearchAtlas/embeddings/bge-small-en-v1.5-v1/manifest.json

SHA-256 dos embeddings:
13fe24f37622790f6d6630fbdb5e91742060f95a8d7310597d3bd98f2a7f465c


In [21]:
import hashlib
import json

import numpy as np
import pandas as pd

saved_embeddings = np.load(EMBEDDINGS_PATH)
saved_metadata = pd.read_parquet(METADATA_PATH)

saved_manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

print("=== VALIDAÇÃO DOS ARTEFATOS ===")
print("Embeddings:", saved_embeddings.shape)
print("Metadados:", saved_metadata.shape)
print("IDs únicos:", saved_metadata["openalex_id"].nunique())

assert saved_embeddings.shape == (5000, 384)
assert len(saved_metadata) == 5000
assert saved_metadata["openalex_id"].is_unique

assert saved_metadata["embedding_index"].tolist() == list(
    range(5000)
)

assert np.array_equal(
    saved_embeddings,
    article_embeddings,
)

assert np.isfinite(saved_embeddings).all()

norms = np.linalg.norm(saved_embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5)

with EMBEDDINGS_PATH.open("rb") as file:
    saved_hash = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

assert saved_hash == saved_manifest["embedding_file_sha256"]
assert saved_manifest["dataset_sha256"] == EXPECTED_SHA256
assert saved_manifest["article_count"] == 5000

print("\n=== INTEGRIDADE ===")
print("Dataset SHA-256:", saved_manifest["dataset_sha256"])
print("Embeddings SHA-256:", saved_hash)
print("Valores finitos:", np.isfinite(saved_embeddings).all())
print("Vetores normalizados:", np.allclose(norms, 1.0, atol=1e-5))

print("\nVALIDAÇÃO FINAL: APROVADA")


=== VALIDAÇÃO DOS ARTEFATOS ===
Embeddings: (5000, 384)
Metadados: (5000, 9)
IDs únicos: 5000

=== INTEGRIDADE ===
Dataset SHA-256: 533fe010fee10dcc87e4ec196886e8a8d9ec6e17f0f4f3d507dc0cf278c79389
Embeddings SHA-256: 13fe24f37622790f6d6630fbdb5e91742060f95a8d7310597d3bd98f2a7f465c
Valores finitos: True
Vetores normalizados: True

VALIDAÇÃO FINAL: APROVADA
